# ARASO Strategy — 5-Year Backtest Analysis

**Date:** 2026-02-16  
**Strategy:** Adaptive Residual Arbitrage with Sentiment Overlay  
**Period:** 2019-01-01 to 2024-12-31 (5 years)  
**Objective:** Validate ARASO + Information Arbitrage with historical data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import sys
sys.path.append('../')

from strategies.araso_backtest import ARASOBacktester, BacktestConfig
from strategies.araso_factor_model import DatabaseConnector

%matplotlib inline
sns.set_style('whitegrid')

## 1. Configuration & Universe Setup

In [ ]:
# Database connection
db = DatabaseConnector(
    username="your_username",
    password="your_password"
)
db.connect()

# Universe selection
# Primary: Tech/SaaS for AI disruption thesis
# Secondary: GPU supply chain for tertiary wave analysis

universe = [
    # Core tech / AI disruptors
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA',
    
    # SaaS vulnerable to AI disruption (Adobe thesis)
    'ADBE', 'ADSK', 'INTU', 'CRM', 'NOW', 'TEAM',
    
    # GPU supply chain (tertiary wave)
    'VRT', 'FN',  # Add more as available
]

print(f"Universe: {len(universe)} stocks")
print(f"Tickers: {', '.join(universe[:10])}...")

## 2. Backtest Configuration

In [ ]:
config = BacktestConfig(
    start_date=datetime(2019, 1, 1),
    end_date=datetime(2024, 12, 31),
    initial_capital=1_000_000,
    rebalance_freq=5,  # Weekly rebalancing
    transaction_cost=0.001,  # 10bps
    max_positions=20,
    max_sector_exposure=0.15,
    max_single_position=0.05,
    stop_loss_pct=0.03,
    entry_zscore=1.5,
    exit_zscore=0.3,
    hmm_threshold=0.7
)

print("Backtest Configuration:")
print(f"  Period: {config.start_date.date()} to {config.end_date.date()}")
print(f"  Initial Capital: ${config.initial_capital:,.0f}")
print(f"  Rebalance: Every {config.rebalance_freq} days")
print(f"  Transaction Cost: {config.transaction_cost*100:.0f}bps")
print(f"  Max Positions: {config.max_positions}")

## 3. Run Backtest

In [ ]:
# Initialize backtester
backtester = ARASOBacktester(db, config)

# Run backtest
results = backtester.run_backtest(universe)

# Print summary
backtester.print_report(results)

## 4. Performance Visualization

In [ ]:
# Convert portfolio history to DataFrame
portfolio_df = pd.DataFrame([
    {
        'date': p.date,
        'total_value': p.total_value,
        'cash': p.cash,
        'daily_return': p.daily_return
    }
    for p in backtester.portfolio_history
]).set_index('date')

# Calculate cumulative returns
portfolio_df['cumulative_return'] = (portfolio_df['total_value'] / config.initial_capital - 1) * 100

# Plot equity curve
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Equity curve
ax1 = axes[0]
ax1.plot(portfolio_df.index, portfolio_df['cumulative_return'], label='ARASO Strategy', linewidth=2)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('ARASO Strategy — Equity Curve (5-Year Backtest)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Cumulative Return (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Rolling Sharpe
ax2 = axes[1]
rolling_sharpe = portfolio_df['daily_return'].rolling(252).mean() / portfolio_df['daily_return'].rolling(252).std() * np.sqrt(252)
ax2.plot(portfolio_df.index, rolling_sharpe, color='green', label='Rolling Sharpe (1Y)', linewidth=2)
ax2.axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='Target Sharpe')
ax2.set_title('Rolling Sharpe Ratio (1-Year Window)', fontsize=14)
ax2.set_ylabel('Sharpe Ratio')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Drawdown
ax3 = axes[2]
cumulative = (1 + portfolio_df['daily_return']).cumprod()
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max * 100
ax3.fill_between(portfolio_df.index, drawdown, 0, color='red', alpha=0.3)
ax3.plot(portfolio_df.index, drawdown, color='red', linewidth=1)
ax3.set_title('Drawdown', fontsize=14)
ax3.set_ylabel('Drawdown (%)')
ax3.set_xlabel('Date')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Trade Analysis

In [ ]:
# Convert trades to DataFrame
trades_df = pd.DataFrame([
    {
        'ticker': t.ticker,
        'entry_date': t.entry_date,
        'exit_date': t.exit_date,
        'direction': 'Long' if t.direction == 1 else 'Short',
        'entry_price': t.entry_price,
        'exit_price': t.exit_price,
        'pnl': t.pnl,
        'exit_reason': t.exit_reason
    }
    for t in backtester.trades if t.pnl is not None
])

# Trade statistics
print("Trade Statistics:")
print(f"  Total Trades: {len(trades_df)}")
print(f"  Winning Trades: {(trades_df['pnl'] > 0).sum()}")
print(f"  Losing Trades: {(trades_df['pnl'] < 0).sum()}")
print(f"  Win Rate: {(trades_df['pnl'] > 0).mean()*100:.1f}%")

# P&L distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(trades_df['pnl'], bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].set_title('Trade P&L Distribution')
axes[0].set_xlabel('P&L ($)')
axes[0].set_ylabel('Frequency')

# Box plot by direction
trades_df.boxplot(column='pnl', by='direction', ax=axes[1])
axes[1].set_title('P&L by Direction')
axes[1].set_xlabel('Direction')
axes[1].set_ylabel('P&L ($)')

plt.tight_layout()
plt.show()

## 6. Monthly Returns Heatmap

In [ ]:
# Calculate monthly returns
portfolio_df['month'] = portfolio_df.index.to_period('M')
monthly_returns = portfolio_df.groupby('month')['daily_return'].apply(lambda x: (1 + x).prod() - 1) * 100

# Convert to DataFrame for heatmap
monthly_df = monthly_returns.to_frame('return')
monthly_df['year'] = monthly_df.index.year
monthly_df['month'] = monthly_df.index.month

# Pivot for heatmap
heatmap_data = monthly_df.pivot(index='year', columns='month', values='return')

# Plot
plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            cbar_kws={'label': 'Monthly Return (%)'})
plt.title('ARASO Strategy — Monthly Returns Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Year')
plt.show()

## 7. Information Decay Analysis (τ metric)

Validate the information arbitrage thesis by measuring how quickly signals decay.

In [ ]:
# Analyze signal decay
from strategies.information_arbitrage import InformationDecayAnalyzer

analyzer = InformationDecayAnalyzer()

# Example: Analyze Adobe-like events in the data
# Look for large single-day moves and measure peer response lag

# TODO: Implement information decay measurement on actual backtest data
print("Information decay analysis placeholder")
print("Would analyze cross-correlation between primary and secondary assets")

## 8. Save Results

In [ ]:
# Save backtest results
backtester.save_results(results, filename='araso_5year_backtest.json')

# Export trades to CSV
trades_df.to_csv('results/araso_trades.csv', index=False)

# Export portfolio history
portfolio_df.to_csv('results/araso_portfolio_history.csv')

print("Results saved to results/ directory")

## 9. Conclusion

**Key Metrics:**
- Sharpe Ratio: {sharpe:.2f}
- Max Drawdown: {max_dd:.1f}%
- Win Rate: {win_rate:.1f}%

**Next Steps:**
1. Parameter sensitivity analysis
2. Walk-forward optimization
3. Out-of-sample validation (2025 data)
4. Paper trading validation